In [1]:
from pathlib import Path

import pandas as pd
from sqlalchemy import text
import matplotlib.pyplot as plt

from data_access.connection import engine


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

## Создание DF на основе данных при инициализации задачи (то есть, момент, когда задача была создана и была заполнена информация о ее приоритете и т.д.)

Проблемой оказалось то, что приоритет на задачу могли установить не в момент ее инициализации, а на последующей итерации

In [2]:
query_path = Path("data_access/queries/training_dataset.sql")
query = text(query_path.read_text(encoding="utf-8"))

issue_df = pd.read_sql_query(query, engine)
issue_df

,ID,Project_ID,Creation_Date,Description_Length,Duration_Days,initial_priority,initial_type
0,85,1,2016-02-24 10:51:02,249,303.1658,Minor,Bug
1,86,1,2016-02-23 18:31:38,125,303.8457,Minor,Improvement
2,89,1,2016-02-22 15:27:31,384,0.1347,Minor,Story
3,90,1,2016-02-17 14:28:31,18,3.4999,Minor,Bug
4,91,1,2016-02-16 14:59:06,579,3.2120,Minor,Improvement
...,...,...,...,...,...,...,...
339241,509031,33,2009-04-09 09:15:20,68,96.3001,Major,Improvement
339242,509032,33,2009-04-09 09:14:27,192,322.5195,Major,Improvement
339243,509033,33,2009-04-09 09:02:06,107,48.2127,Minor - P4,Bug
339244,509034,33,2009-04-09 08:59:26,0,138.1045,Major - P3,Improvement


## Проверка структуры итогового датасета

In [3]:
structure_summary = pd.Series({
    "row_count": len(issue_df),
    "unique_id_count": issue_df["ID"].nunique(),
    "duplicate_id_count": issue_df["ID"].duplicated().sum(),
})

structure_summary

row_count             339246
unique_id_count       339246
duplicate_id_count         0
dtype: int64

In [4]:
missing_values = issue_df.isna().sum().sort_values(ascending=False)

missing_values

ID                    0
Project_ID            0
Creation_Date         0
Description_Length    0
Duration_Days         0
initial_priority      0
initial_type          0
dtype: int64

In [5]:
duration_quality = pd.Series({
    "negative_duration_count": issue_df["Duration_Days"].lt(0).sum(),
    "zero_duration_count": issue_df["Duration_Days"].eq(0).sum(),
    "missing_duration_count": issue_df["Duration_Days"].isna().sum(),
})

duration_quality

negative_duration_count    0
zero_duration_count        0
missing_duration_count     0
dtype: int64

In [6]:
issue_df["Duration_Days"].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
)

count    339246.000000
mean        338.448577
std         591.624797
min           0.003500
50%          36.945150
90%        1230.229450
95%        1769.904025
99%        2552.910410
max        2921.985100
Name: Duration_Days, dtype: float64

In [7]:
category_summary = issue_df[[
    "Project_ID",
    "initial_type",
    "initial_priority",
]].nunique()

category_summary

Project_ID          39
initial_type        71
initial_priority    19
dtype: int64

In [8]:
issue_df["initial_priority"].value_counts(dropna=False)

initial_priority
Minor             66068
Major - P3        51327
Unknown           49664
Low               43223
Medium            41491
Major             32540
None              14651
High              10585
To be reviewed     9230
Critical           5292
Minor - P4         4228
Trivial            3058
Blocker            2788
Highest            1701
Critical - P2      1602
Trivial - P5        745
Blocker - P1        589
Undefined           334
Lowest              130
Name: count, dtype: int64

In [9]:
issue_df["initial_type"].value_counts(dropna=False)

initial_type
Bug            176049
Improvement     50701
Task            29457
Story           28294
New Feature     18598
                ...  
Paper               1
Action Item         1
IT Help             1
Workload            1
Change              1
Name: count, Length: 71, dtype: int64

## Хронологическое разделение данных

In [10]:
issue_df = issue_df.sort_values(
    ["Creation_Date", "ID"]
).reset_index(drop=True)

train_end = int(len(issue_df) * 0.70)
validation_end = int(len(issue_df) * 0.85)

train_df = issue_df.iloc[:train_end].copy()
validation_df = issue_df.iloc[train_end:validation_end].copy()
test_df = issue_df.iloc[validation_end:].copy()

In [11]:
split_summary = pd.DataFrame({
    "train": {
        "row_count": len(train_df),
        "creation_date_min": train_df["Creation_Date"].min(),
        "creation_date_max": train_df["Creation_Date"].max(),
    },
    "validation": {
        "row_count": len(validation_df),
        "creation_date_min": validation_df["Creation_Date"].min(),
        "creation_date_max": validation_df["Creation_Date"].max(),
    },
    "test": {
        "row_count": len(test_df),
        "creation_date_min": test_df["Creation_Date"].min(),
        "creation_date_max": test_df["Creation_Date"].max(),
    },
}).T

split_summary

,row_count,creation_date_min,creation_date_max
train,237472,2002-04-24 02:39:37,2016-09-30 09:15:12
validation,50887,2016-09-30 09:15:14,2018-06-06 16:04:23
test,50887,2018-06-06 16:12:09,2020-10-22 15:16:24


In [12]:
duration_distribution = pd.DataFrame({
    "train": train_df["Duration_Days"].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    ),
    "validation": validation_df["Duration_Days"].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    ),
    "test": test_df["Duration_Days"].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    ),
}).T

duration_distribution

,count,mean,std,min,50%,90%,95%,99%,max
train,237472.0,427.537595,670.075521,0.0035,51.7338,1510.60861,1995.107195,2638.903067,2921.9851
validation,50887.0,195.781059,304.558674,0.0035,33.7652,705.13046,929.392310,1130.765630,1476.3147
test,50887.0,65.368513,115.806334,0.0035,17.8897,192.09910,344.054370,546.540272,850.7518
